# K06_04 – XGBoost auf Iris als Praxisausblick (Student)

## Lernziele
- Entscheidungsbaum, Random Forest und XGBoost vergleichen
- verstehen, warum ein einzelner Testsplit kein verlässlicher Benchmark ist
- Cross-Validation als robustere Bewertungsbasis einordnen
- Leistung und Interpretierbarkeit gemeinsam diskutieren

**Wichtig:**  
Dieses Notebook ist als **Praxisausblick** gedacht.  
Auf dem kleinen Iris-Datensatz sind Unterschiede zwischen den Verfahren oft relativ klein.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except Exception:
    XGBOOST_AVAILABLE = False
    print("Hinweis: xgboost ist in dieser Umgebung nicht installiert.")

## 1. Iris-Daten laden

In [2]:
iris = load_iris()
X = iris.data
y = iris.target
target_names = iris.target_names

## 2. Trainings- und Testdaten

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

## 3. Modelle definieren

In [4]:
models = [
    ("Entscheidungsbaum", DecisionTreeClassifier(random_state=42)),
    ("Random Forest", RandomForestClassifier(n_estimators=150, random_state=42))
]

if XGBOOST_AVAILABLE:
    models.append((
        "XGBoost",
        XGBClassifier(
            n_estimators=60,
            max_depth=3,
            learning_rate=0.1,
            subsample=1.0,
            colsample_bytree=1.0,
            objective="multi:softprob",
            eval_metric="mlogloss",
            random_state=42,
            n_jobs=1,
            verbosity=0
        )
    ))

## 4. Einzelner Train/Test-Split: erster Vergleich

In [5]:
split_results = []
for name, clf in models:
    clf.fit(X_train, y_train)
    split_results.append({
        "Verfahren": name,
        "Train-Accuracy": clf.score(X_train, y_train),
        "Test-Accuracy": clf.score(X_test, y_test)
    })

split_df = pd.DataFrame(split_results).round(4)
split_df

,Verfahren,Train-Accuracy,Test-Accuracy
0,Entscheidungsbaum,1.0000,0.9333
1,Random Forest,1.0000,0.8889
2,XGBoost,0.9905,0.9333


## 5. Warum dieser Vergleich vorsichtig gelesen werden muss

Bei einem kleinen und relativ einfachen Datensatz wie **Iris** kann es passieren, dass ein einzelner Entscheidungsbaum auf einem konkreten Split sehr gut abschneidet – manchmal sogar besser als ein Random Forest.

Das ist **nicht** automatisch ein Widerspruch zur allgemeinen Modelllogik.

Wichtig ist:

- Ein einzelner Split ist zufallsabhängig.
- Die Iris-Daten sind klein und vergleichsweise gut trennbar.
- Ein Entscheidungsbaum kann auf einem bestimmten Split zufällig sehr gut passen.
- Random Forest und XGBoost sind oft nicht deshalb interessant, weil sie **immer auf jedem Einzelwert** besser sind, sondern weil sie im Mittel **robuster** oder **leistungsstärker** sein können.

Deshalb ergänzen wir jetzt eine **Cross-Validation**.

**Didaktischer Hinweis:**  
Dieses Notebook ist eher ein **Praxisausblick auf Modellfamilien** als ein harter Leistungsvergleich.

In [6]:
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

cv_results = []
for name, clf in models:
    scores = cross_validate(
        clf,
        X,
        y,
        cv=cv,
        scoring="accuracy",
        return_train_score=True
    )
    cv_results.append({
        "Verfahren": name,
        "CV-Accuracy Mittelwert": scores["test_score"].mean(),
        "CV-Accuracy Std": scores["test_score"].std(),
        "Train-Accuracy Mittelwert": scores["train_score"].mean()
    })

cv_df = pd.DataFrame(cv_results).round(4)
cv_df

,Verfahren,CV-Accuracy Mittelwert,CV-Accuracy Std,Train-Accuracy Mittelwert
0,Entscheidungsbaum,0.9333,0.0516,1.0000
1,Random Forest,0.9533,0.0521,1.0000
2,XGBoost,0.9533,0.0521,0.9993


## 6. Interpretation der Cross-Validation

Lesen Sie die Cross-Validation bitte so:

- **CV-Accuracy Mittelwert**: Wie gut ist das Verfahren im Mittel über mehrere Aufteilungen?
- **CV-Accuracy Std**: Wie stark schwanken die Ergebnisse?
- **Train-Accuracy Mittelwert**: Gibt es Hinweise auf Overfitting?

**Wichtiger Punkt für die Vorlesung:**  
Wenn der Entscheidungsbaum in einem einzelnen Split einmal besser aussieht, heißt das nicht automatisch, dass er das bessere Verfahren ist.  
Für die Vorlesung ist die **Cross-Validation-Tabelle** die verlässlichere Grundlage.

## 7. Interpretation: Leistung und Interpretierbarkeit

Nicht nur die Accuracy ist wichtig.

Auch folgende Fragen zählen:
- Wie gut ist das Verfahren **erklärbar**?
- Wie stark schwanken die Ergebnisse?
- Wie aufwendig ist das Training?
- Wie wichtig ist maximale Leistung im Vergleich zur Nachvollziehbarkeit?

Typische Einordnung:
- **Entscheidungsbaum**: gut interpretierbar
- **Random Forest**: robuster, aber weniger transparent
- **XGBoost**: oft leistungsstark, aber deutlich schwerer zu interpretieren

## 8. Mini-Übungen

1. Warum reicht ein einzelner Testsplit nicht aus, um drei Verfahren fair zu vergleichen?
2. Warum sollte man die Ergebnisse auf Iris nicht überinterpretieren?
3. Welches Verfahren wirkt in der Cross-Validation am stabilsten?

## 9. Fazit

- Auf kleinen Datensätzen können Einzelsplits täuschen.
- Ein einzelner Entscheidungsbaum kann zufällig sehr gut abschneiden.
- Für einen belastbaren Vergleich ist die **Cross-Validation** besser geeignet.
- Modellwahl ist immer auch ein Abwägen von **Leistung**, **Stabilität** und **Interpretierbarkeit**.